# Aula 3: Boosting com XGBoost

Esta aula aborda a técnica de **Boosting** utilizando o algoritmo **XGBoost** (Extreme Gradient Boosting). O boosting é um método de ensemble que treina modelos sequencialmente, onde cada novo modelo corrige os erros do anterior. O XGBoost é uma implementação otimizada de gradient boosting que oferece alta performance e escalabilidade.

**Conceitos-chave:**
- **Boosting**: Combina modelos fracos (geralmente árvores de decisão rasas) em um modelo forte, treinando sequencialmente e dando mais peso aos exemplos mal classificados.
- **XGBoost**: Implementação eficiente com regularização (L1/L2), tratamento de valores ausentes, paralelização e early stopping.
- **Objective**: Para classificação binária, usa `binary:logistic` que retorna probabilidades.
- **Learning rate (eta)**: Controla a contribuição de cada árvore; valores menores exigem mais árvores mas generalizam melhor.
- **Early stopping**: Interrompe o treino quando a métrica de validação não melhora, evitando overfitting.

In [ ]:
# Importações
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)


In [ ]:
# Definição do random state
RANDOM_STATE = 7895


## 1. Carregamento dos Dados

Carregamos o dataset **BankChurners** (clientes de banco com indicação de churn) diretamente de um repositório GitHub. O dataset contém variáveis demográficas, comportamentais e a variável alvo `Target` indicando se o cliente está ativo ou inativo.

In [ ]:
# 1. Carregando os dados
df = pd.read_csv('https://raw.githubusercontent.com/vqrca/ml-datasets/refs/heads/main/BankChurners_portugues.csv')


In [ ]:
# Exibição das primeiras observações para verificar a estrutura dos dados e as variáveis disponíveis
pd.set_option('display.max_columns', None)
print(df.head())


In [ ]:
# Checando detalhes sobre os dados
df.info()


## 2. Preparação dos Dados: Variáveis Categóricas

O XGBoost nativo (versão 3.3.0) aceita variáveis do tipo `category` do pandas, o que evita a necessidade de one-hot encoding manual. Convertemos todas as colunas do tipo `object` para `category`. Isso permite que o algoritmo trate as categorias de forma nativa e eficiente.

In [ ]:
# 2. Preparando os dados
colunas_categoricas = df.select_dtypes(include='object').columns

df[colunas_categoricas] = df[colunas_categoricas].astype('category')


In [ ]:
# Verificando a alteração no tipo de dado object para category
df.info()


## 3. Separação em Treino e Teste

Separamos as features (`X`) da variável alvo (`y`). Removemos `Numero_Cliente` (identificador) e `Target` (alvo). Em seguida, fazemos o split estratificado (mantém a proporção das classes) em 80% treino e 20% teste.

In [ ]:
# 3. Separando os dados em treino e teste

X = df.drop(['Numero_Cliente', 'Target'], axis=1)
y = df['Target']


In [ ]:
X.head()


In [ ]:
y.head()


### Preparação do Target para XGBoost

O XGBoost espera que o target binário seja numérico (0 e 1). Usamos `LabelEncoder` para transformar as labels categóricas ('Ativo'/'Inativo') em 0 e 1.

In [ ]:
# Preparando o Target para o XGBoost
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df['Target'])


In [ ]:
y


In [ ]:
# Separação dos dados em conjuntos de treinamento e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)


## 4. Treinamento Inicial com XGBoost

Instanciamos um `XGBClassifier` com `objective='binary:logistic'` para classificação binária. O modelo é treinado nos dados de treino e faz predições nos dados de teste.

In [ ]:
# 4. Treinamento com XGBoost
import xgboost as xgb

modelo_xgboost = xgb.XGBClassifier(objective='binary:logistic',
                                   random_state=RANDOM_STATE
)
modelo_xgboost.fit(X_treino, y_treino)
predicoes = modelo_xgboost.predict(X_teste)


## 5. Avaliação do Modelo: Métricas de Performance

Comparamos a acurácia entre treino e teste para detectar **overfitting**. Se a acurácia de treino for muito superior à de teste, o modelo está decorando os dados de treino.

Também geramos:
- **Classification report**: Precisão, revocação (recall) e F1-score por classe.
- **Matriz de confusão normalizada**: Visualiza acertos/erros por classe.
- **Curva ROC e AUC**: Avalia a capacidade discriminativa do modelo em diferentes thresholds.

In [ ]:
# Análise da acurácia: Comparação da acurácia entre treino e teste: verificando o overfitting
# Acurácia no teste
acuracia_teste = accuracy_score(y_teste, predicoes)
print(f'Acurácia do XGBoost nos dados de teste: {acuracia_teste:.2f}')


In [ ]:
# Acurácia no treino
predicoes_treino = modelo_xgboost.predict(X_treino)
acuracia = accuracy_score(y_treino, predicoes_treino)
print(f'Acurácia do XGBoost nos dados de treino: {acuracia:.2f}')


In [ ]:
# Relatório de métricas: Análise detalhada por classe: precisão, revocação e F1-score
report = classification_report(y_teste, predicoes)
print(report)


In [ ]:
# Matriz de confusão: Distribuição dos acertos e erros por classe
ConfusionMatrixDisplay.from_estimator(modelo_xgboost,
                                       X_teste, y_teste,
                                       normalize='true',
                                       cmap='Blues',
                                       display_labels=['Ativo', 'Inativo'])


In [ ]:
# Curva ROC: Análise da capacidade discriminativa: curva ROC e AUC
# Gera a curva ROC
RocCurveDisplay.from_estimator(
    modelo_xgboost,
    X_teste,
    y_teste
)


## 6. Validação Cruzada com XGBoost

A validação cruzada (cross-validation) avalia a generalização do modelo dividindo os dados em *k* folds e treinando/validando *k* vezes. Usamos `xgb.cv` que trabalha com `DMatrix` (estrutura de dados otimizada do XGBoost).

Aqui usamos 3 folds e apenas 5 rodadas de boosting (`num_boost_round=5`) para demonstração rápida.

In [ ]:
# 5. Realizando a validação cruzada
dmatrix = xgb.DMatrix(data=X, label=y)

params = {'objective': 'binary:logistic'}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=5, metrics='error',
                       as_pandas=True, seed=5678)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


## 7. Early Stopping

O **early stopping** monitora a métrica de validação a cada iteração e para o treino quando não há melhora por `early_stopping_rounds` rodadas consecutivas. Isso evita overfitting e economiza tempo computacional.

Comparamos:
- Sem early stopping (200 rodadas fixas)
- Com early stopping (para automaticamente)

In [ ]:
# Explorando a técnica de Early Stopping
params = {'objective': 'binary:logistic'}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=200, metrics='error',
                       as_pandas=True, seed=5678)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


In [ ]:
# early_stopping_rounds=5
params = {'objective': 'binary:logistic'}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=200, early_stopping_rounds=5,
                       metrics='error', as_pandas=True, seed=5678)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


## 8. Taxa de Aprendizado (Learning Rate)

O parâmetro `learning_rate` (ou `eta`) controla o tamanho do passo na direção do gradiente. Valores menores (ex: 0.01) tornam o aprendizado mais lento mas mais robusto, exigindo mais árvores (`n_estimators` ou `num_boost_round`). Valores maiores (ex: 1) aprendem rápido mas podem divergir ou overfitar.

Testamos três valores: 0.01, 0.1 e 1.0, mantendo 100 rodadas de boosting.

In [ ]:
# 6. Analisando a taxa de aprendizado
params = {'objective': 'binary:logistic',
          'learning_rate': 0.01}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=100,
                       metrics='error', as_pandas=True, seed=123)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


In [ ]:
# learning_rate = 0.1
params = {'objective': 'binary:logistic',
          'learning_rate': 0.1}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=100,
                       metrics='error', as_pandas=True, seed=123)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


In [ ]:
# learning_rate = 1
params = {'objective': 'binary:logistic',
          'learning_rate': 1}

cv_resultados = xgb.cv(dtrain=dmatrix, params=params, nfold=3,
                       num_boost_round=100,
                       metrics='error', as_pandas=True, seed=123)

print(cv_resultados)
acuracia = 1 - cv_resultados['test-error-mean'].iloc[-1]
print(f'Acuracia: {acuracia}')


## 9. Busca Aleatória de Hiperparâmetros (RandomizedSearchCV)

O XGBoost possui muitos hiperparâmetros que afetam performance:
- `learning_rate`: taxa de aprendizado
- `max_depth`: profundidade máxima das árvores (controla complexidade)
- `colsample_bytree`: fração de features usadas por árvore (regularização)
- `n_estimators`: número de árvores
- `reg_lambda` / `reg_alpha`: regularização L2 / L1 nos pesos
- `gamma`: ganho mínimo para fazer um split
- `min_child_weight`: soma mínima de pesos (hessianos) em um nó filho

Usamos `RandomizedSearchCV` do scikit-learn com 5 iterações e 4-fold CV, otimizando **recall** (importante para detecção de churn).

In [ ]:
# 7. Ajustando hiperparâmetros
params = {
    'learning_rate': [0.1, 0.2, 0.3],
    'max_depth': [5, 10, 15],
    'colsample_bytree': [0.5, 1.0],
    'n_estimators': [50, 100],
    'reg_lambda': [1, 5, 10],
    'reg_alpha':  [0, 0.1, 1],
    'gamma':      [0, 0.1, 1],
    'min_child_weight': [1, 5]
}

modelo_xgb = xgb.XGBClassifier(random_state=RANDOM_STATE)

random_search_cv = RandomizedSearchCV(estimator=modelo_xgb, param_distributions=params, n_iter=5, cv=4, scoring='recall',
                                      verbose=1, random_state=765)

random_search_cv.fit(X,y)

print('Melhores parametros encontrados: ', random_search_cv.best_params_)
print('Recall:', random_search_cv.best_score_)


## 10. Avaliação Final do Modelo Ajustado

Avaliamos o melhor modelo encontrado na busca aleatória nos dados de teste (não vistos durante a busca) e comparamos com o treino para verificar overfitting. Geramos novamente classification report, matriz de confusão e curva ROC.

In [ ]:
# Analisando as métricas no teste após ajuste dos hiperparâmetros
modelo_xgboost_ajustado = random_search_cv.best_estimator_
predicoes_xgboost_ajustado = modelo_xgboost_ajustado.predict(X_teste)
acuracia = accuracy_score(y_teste, predicoes_xgboost_ajustado)
print(f"Acurácia do XGBoost ajustado nos dados de teste: {acuracia:.2%}")


In [ ]:
# Analisando as métricas no treino após ajuste dos hiperparâmetros
predicoes_xgboost_ajustado_treino = modelo_xgboost_ajustado.predict(X_treino)
acuracia = accuracy_score(y_treino, predicoes_xgboost_ajustado_treino)
print(f"Acurácia do XGBoost ajustado nos dados de treino: {acuracia:.2%}")


In [ ]:
# Classification report
report = classification_report(y_teste, predicoes_xgboost_ajustado)
print(report)


In [ ]:
# Matriz de confusão
ConfusionMatrixDisplay.from_estimator(modelo_xgboost_ajustado,
                                       X_teste, y_teste,
                                       normalize='true', cmap='Blues',
                                       display_labels=['Ativo', 'Inativo'])


In [ ]:
# Curva ROC
RocCurveDisplay.from_predictions(y_teste, predicoes_xgboost_ajustado,
                                 name='XGBoost ajustado')


## 11. Importância das Variáveis

O XGBoost fornece `feature_importances_` baseado no ganho (gain) médio de cada feature nos splits das árvores. Features com maior importância contribuem mais para as predições. Isso auxilia na interpretabilidade e seleção de features.

In [ ]:
# Importância das variáveis
importancias = (
    pd.Series(
        modelo_xgboost_ajustado.feature_importances_,
        index=X_treino.columns
    )
    .sort_values(ascending=False)
)

print(importancias.head(15))
